# 03 · Rolling Correlations & Structural Breaks
**Brazilian Stock-Bond Correlation Study**

This notebook produces the **headline time-series chart** of the paper — the rolling
Ibovespa × bond correlation. This is the Brazilian equivalent of the IMF's Figure 1.

1. 252-day rolling Pearson and Spearman correlations
2. Conditional correlation: ρ given equity in bottom 10th percentile
3. CUSUM structural break test
4. Regime-average correlation summary

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as scipy_stats
import statsmodels.api as sm

from fetch import load_master, CRISES, REGIMES

master = load_master()

plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
})
CRISIS_COLORS = {
    "GFC":"#d62728","Dilma":"#ff7f0e","Joesley":"#9467bd",
    "COVID":"#2ca02c","Americanas":"#8c564b","Fiscal24":"#e377c2",
}
LABELS = {
    "ibov":"Ibovespa", "ntnb":"NTN-B 5y",
    "ltn":"LTN 2y", "ntnf":"NTN-F 10y", "lft":"LFT 1y",
}

def add_crisis_bands(ax, alpha=0.15):
    for name, (s, e) in CRISES.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e),
                   color=CRISIS_COLORS[name], alpha=alpha, label=name)

## 1. The headline chart: 252-day rolling correlation

**This is Figure 4 of the whitepaper.**
It directly replicates and extends the IMF's approach, showing Brazil's
correlation dynamics over two decades.

In [ ]:
WINDOW = 252  # 1 trading year
bond_cols = ["ntnb", "ltn", "ntnf", "lft"]
bond_colors = ["#d62728", "#ff7f0e", "#2ca02c", "#9467bd"]

df_ret = master[["ibov"] + bond_cols].dropna(how="all")

# Rolling Pearson correlations
roll_corr = pd.DataFrame({
    col: df_ret["ibov"].rolling(WINDOW).corr(df_ret[col])
    for col in bond_cols
})

fig, ax = plt.subplots(figsize=(14, 5.5))
for col, color in zip(bond_cols, bond_colors):
    ax.plot(roll_corr.index, roll_corr[col],
            label=LABELS[col], lw=1.5, color=color)

ax.axhline(y=0, color="black", lw=1.2, ls="--", alpha=0.7, label="ρ = 0")
add_crisis_bands(ax, alpha=0.13)

# Highlight the IMF's 2019 turning point for advanced economies
ax.axvline(pd.Timestamp("2020-01-01"), color="navy", lw=1.5, ls=":",
           alpha=0.8, label="IMF regime shift (DM, 2020)")

ax.set_ylim(-0.7, 0.8)
ax.set_ylabel(f"Rolling {WINDOW}-day Pearson ρ", fontsize=11)
ax.set_title(
    f"Ibovespa vs. Brazilian bond indices: {WINDOW}-day rolling correlation\n"
    "Brazil, 2005–2026",
    fontsize=13,
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

# Build legend (assets + crises)
asset_handles = [plt.Line2D([0],[0], color=c, lw=2, label=LABELS[col])
                 for col, c in zip(bond_cols, bond_colors)]
asset_handles.append(plt.Line2D([0],[0], color="black", lw=1.5,
                                 ls="--", label="ρ = 0"))
asset_handles.append(plt.Line2D([0],[0], color="navy", lw=1.5,
                                 ls=":", label="IMF regime shift (DM)"))
crisis_handles = [plt.Rectangle((0,0),1,1, fc=CRISIS_COLORS[n],
                                  alpha=0.4, label=n)
                  for n in CRISES]
ax.legend(handles=asset_handles + crisis_handles,
          loc="lower left", fontsize=8, ncol=3)

plt.tight_layout()
plt.savefig("../outputs/fig_rolling_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_rolling_correlation.png")

## 2. Tail conditional correlation — ρ given equity stress

The key question: **do bonds provide diversification when equity markets are crashing?**

We compute the correlation of bond returns conditional on equity returns being in the
bottom 10th percentile of observations — the "stress correlation."

In [ ]:
RET_COLS_BONDS = ["ntnb", "ltn", "ntnf", "lft"]
df_ret = master[["ibov"] + RET_COLS_BONDS].dropna(how="all") * 100

q10 = df_ret["ibov"].quantile(0.10)
q25 = df_ret["ibov"].quantile(0.25)

stress_mask_10 = df_ret["ibov"] <= q10  # bottom 10%
stress_mask_25 = df_ret["ibov"] <= q25  # bottom 25%

results = []
for col in RET_COLS_BONDS:
    pair = df_ret[["ibov", col]].dropna()
    r_full    = pair["ibov"].corr(pair[col])
    r_stress10 = pair.loc[stress_mask_10, "ibov"].corr(pair.loc[stress_mask_10, col])
    r_stress25 = pair.loc[stress_mask_25, "ibov"].corr(pair.loc[stress_mask_25, col])
    results.append({
        "Bond": LABELS[col],
        "ρ full sample":    round(r_full, 3),
        "ρ | equity < Q10": round(r_stress10, 3),
        "ρ | equity < Q25": round(r_stress25, 3),
        "Δ (stress–full)":  round(r_stress10 - r_full, 3),
    })

cond_df = pd.DataFrame(results).set_index("Bond")
print("=== Conditional tail correlations ===")
print("(Positive Δ = correlations INCREASE during equity stress)")
print(cond_df.to_string())
cond_df.to_csv("../outputs/nb_tbl_conditional_correlations.csv")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))
x  = np.arange(len(cond_df))
w  = 0.28
b1 = ax.bar(x - w, cond_df["ρ full sample"],    w, label="Full sample",     color="#1f77b4")
b2 = ax.bar(x,     cond_df["ρ | equity < Q25"], w, label="Equity < Q25 %",  color="#ff7f0e")
b3 = ax.bar(x + w, cond_df["ρ | equity < Q10"], w, label="Equity < Q10 %",  color="#d62728")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_xticks(x); ax.set_xticklabels(cond_df.index, fontsize=10)
ax.set_ylabel("Pearson ρ with Ibovespa")
ax.set_title("Conditional tail correlations: Ibovespa vs. bond classes\n"
             "(Diversification works only if bars are negative during equity stress)",
             fontsize=11)
ax.legend(fontsize=9)
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}", xy=(bar.get_x()+bar.get_width()/2, h),
                    xytext=(0, 3 if h >= 0 else -10),
                    textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("../outputs/fig_conditional_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. CUSUM structural break test

The CUSUM test on the rolling Ibovespa × NTN-B correlation identifies
statistically significant regime shifts. This is a simpler alternative
to Bai-Perron (which requires R via rpy2) and sufficient for whitepaper evidence.

In [ ]:
from statsmodels.stats.diagnostic import breaks_cusumolsresid
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# Run OLS of ibov on ntnb returns, then CUSUM on residuals
df_pair = master[["ibov", "ntnb"]].dropna() * 100
y = df_pair["ibov"].values
X = add_constant(df_pair["ntnb"].values)

ols_res = OLS(y, X).fit()

# breaks_cusumolsresid returns THREE values -- (statistic, p-value, critical values) --
# not a CUSUM path. The statistic is sup|W(t)| for the OLS-CUSUM process
#     W(t) = (1 / (sigma_hat * sqrt(n))) * sum_{i<=nt} e_i,
# so the path has to be built explicitly to plot it against its bands.
stat, pval, crit_vals = breaks_cusumolsresid(ols_res.resid)
crit = dict((lvl, c) for lvl, c in crit_vals)[5]        # 5% band for sup|W(t)|

resid = ols_res.resid
n     = len(resid)
sigma = np.sqrt(np.sum(resid**2) / (n - X.shape[1]))
cusum = np.cumsum(resid) / (sigma * np.sqrt(n))         # W(t), asymptotically a Brownian bridge
dates_cusum = df_pair.index

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# CUSUM path against its 5% boundary
ax = axes[0]
ax.plot(dates_cusum, cusum, color="#1f77b4", lw=1.5, label="OLS-CUSUM path W(t)")
ax.axhline( crit, color="#d62728", ls="--", lw=1.2, label=f"5% band (+/-{crit})")
ax.axhline(-crit, color="#d62728", ls="--", lw=1.2)
ax.axhline(0, color="black", lw=0.6)
for name, (s, e) in CRISES.items():
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e),
               color=CRISIS_COLORS[name], alpha=0.12)
ax.set_ylabel("CUSUM")
ax.set_title("CUSUM test: structural stability of Ibovespa ~ NTN-B OLS relationship",
             fontsize=12)
ax.legend(fontsize=9)

# Rolling 63-day correlation alongside
ax2 = axes[1]
rc63 = master["ibov"].rolling(63).corr(master["ntnb"])
ax2.plot(rc63.index, rc63, color="#ff7f0e", lw=1.2, alpha=0.8, label="63-day rolling ρ")
rc252 = master["ibov"].rolling(252).corr(master["ntnb"])
ax2.plot(rc252.index, rc252, color="#2ca02c", lw=1.8, label="252-day rolling ρ")
ax2.axhline(0, color="black", ls="--", lw=0.8)
for name, (s, e) in CRISES.items():
    ax2.axvspan(pd.Timestamp(s), pd.Timestamp(e),
                color=CRISIS_COLORS[name], alpha=0.12)
ax2.set_ylabel("Pearson ρ")
ax2.set_title("Rolling correlation: Ibovespa vs. NTN-B 5y", fontsize=12)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../outputs/fig_cusum_break_test.png", dpi=150, bbox_inches="tight")
plt.show()

exceeds = bool(np.max(np.abs(cusum)) > crit)
print("=== OLS-CUSUM test (Brown, Durbin & Evans 1975) ===")
print(f"  H0: the Ibovespa ~ NTN-B relationship is stable over the sample")
print(f"  sup|W(t)|      : {np.max(np.abs(cusum)):.3f}   (statsmodels: {stat:.3f})")
print(f"  p-value        : {pval:.3f}")
print(f"  5% band        : +/-{crit}")
print(f"  crosses band   : {exceeds}")
print()
if pval < 0.05:
    print("  -> Rejects stability: there is evidence of a structural break.")
else:
    print("  -> Does NOT reject stability at the 5% level.")
    print("     Note what this does and does not say. Failing to reject is not proof")
    print("     of a stable relationship, and it is not inconsistent with the regime")
    print("     variation in the correlations above: CUSUM has low power against slow")
    print("     drift, and this regression is dominated by equity variance because")
    print("     Ibovespa volatility is roughly 5x NTN-B volatility.")

## 4. Spearman vs Pearson: is the relationship monotonic?

Divergence between Pearson and Spearman rolling correlations suggests
**nonlinear** dependence — motivating the copula analysis in notebook 05.

In [ ]:
df_pair = master[["ibov", "ntnb"]].dropna()
W = 252

pearson_roll  = df_pair["ibov"].rolling(W).corr(df_pair["ntnb"])
# Spearman via rank transform
rank_ibov = df_pair["ibov"].rolling(W).rank()
rank_ntnb = df_pair["ntnb"].rolling(W).rank()
spearman_roll = rank_ibov.rolling(1).corr(rank_ntnb)   # already ranked, so Pearson=Spearman
# Cleaner: compute Spearman properly
def rolling_spearman(x, y, w):
    result = pd.Series(index=x.index, dtype=float)
    for i in range(w, len(x)):
        xi = x.iloc[i-w:i]
        yi = y.iloc[i-w:i]
        result.iloc[i] = scipy_stats.spearmanr(xi, yi)[0]
    return result

print("Computing rolling Spearman (this may take ~30s)...")
spearman_roll = rolling_spearman(df_pair["ibov"], df_pair["ntnb"], W)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(pearson_roll.index,  pearson_roll,  lw=1.5, color="#1f77b4", label="Pearson ρ")
ax.plot(spearman_roll.index, spearman_roll, lw=1.5, color="#d62728",
        ls="--", alpha=0.8, label="Spearman ρ")
ax.axhline(0, color="black", lw=0.8, ls="--")
add_crisis_bands(ax, alpha=0.12)
ax.set_title(f"Pearson vs Spearman {W}-day rolling correlation: "
             f"Ibovespa vs NTN-B\n"
             "Divergence = nonlinear dependence → motivates copula analysis",
             fontsize=11)
ax.set_ylabel("Correlation coefficient")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("../outputs/fig_pearson_vs_spearman.png", dpi=150, bbox_inches="tight")
plt.show()

diff = (pearson_roll - spearman_roll).abs()
print(f"Mean |Pearson - Spearman|: {diff.mean():.4f}")
print(f"Max  |Pearson - Spearman|: {diff.max():.4f}")
print("(Large differences indicate nonlinear/asymmetric dependence)")

## 5. Regime-average correlation table

Summary table for the whitepaper — replaces the IMF's cross-country comparison
with Brazil's regime comparison.

In [ ]:
pairs = [("ibov","ntnb"), ("ibov","ltn"), ("ibov","ntnf"), ("ibov","lft")]
pair_labels = {
    ("ibov","ntnb"):      "Ibovespa × NTN-B",
    ("ibov","ltn"):       "Ibovespa × LTN",
    ("ibov","ntnf"):      "Ibovespa × NTN-F",
    ("ibov","lft"): "Ibovespa × LFT",
}

from metrics import corr_with_ci, bootstrap_corr_diff

rows = {}
for name, (s, e) in list(REGIMES.items()) + [("Full sample", ("2004-01-01","2026-12-31"))]:
    sub = master[(master.index >= s) & (master.index <= e)]
    row = {}
    for a, b in pairs:
        r = corr_with_ci(sub[a], sub[b])
        row[pair_labels[(a,b)]]        = round(r["rho"], 3) if np.isfinite(r["rho"]) else np.nan
        row[pair_labels[(a,b)] + " CI"] = (f"[{r['lo']:+.3f},{r['hi']:+.3f}]"
                                           if np.isfinite(r["rho"]) else "")
        row[pair_labels[(a,b)] + " sig"] = "*" if r["sig"] else ""
    row["n"] = len(sub[["ibov","ntnb"]].dropna())
    rows[name] = row

regime_corr_tbl = pd.DataFrame(rows).T
print("=== Regime correlations with 95% Fisher-z confidence intervals ===")
print("(* = interval excludes zero)\n")
for name in regime_corr_tbl.index:
    r = regime_corr_tbl.loc[name]
    print(f"{name:<22} n={int(r['n']):>5}  " +
          "   ".join(f"{lbl.split(' × ')[1]}: {r[lbl]:+.3f} {r[lbl+' CI']}{r[lbl+' sig']}"
                     for lbl in pair_labels.values()))
regime_corr_tbl.to_csv("../outputs/nb_tbl_regime_correlations.csv")
print("\nSaved: outputs/nb_tbl_regime_correlations.csv")

## 6. Are the regime differences statistically distinguishable?

A table of point estimates ordered from low to high invites a story about regime
change. Before telling it, test whether the ordering survives sampling error.

Each regime holds 700–1,300 daily observations, so a correlation has a standard error
of roughly 0.03. Differences smaller than about 0.08 are not distinguishable from
noise, however clean the ranking looks. We use a stationary block bootstrap
(21-day blocks) so the test is not inflated by the serial dependence in daily returns.

In [ ]:
base = "Lula Boom"          # the earliest, and on point estimates the calmest, regime
bs, be = REGIMES[base]
b_sub = master[(master.index >= bs) & (master.index <= be)][["ibov","ntnb"]].dropna()

rows = []
for name, (s, e) in REGIMES.items():
    if name == base:
        continue
    sub = master[(master.index >= s) & (master.index <= e)][["ibov","ntnb"]].dropna()
    t = bootstrap_corr_diff(sub["ibov"], sub["ntnb"],
                            b_sub["ibov"], b_sub["ntnb"], n_boot=1500, block=21)
    rows.append({"Regime": name,
                 f"rho - rho({base})": round(t["diff"], 3),
                 "bootstrap SE": round(t["boot_se"], 3),
                 "p-value": round(t["p"], 3),
                 "differs at 5%": "yes" if t["p"] < 0.05 else "no"})

diff_tbl = pd.DataFrame(rows).set_index("Regime")
print(f"=== Regime correlation vs {base}: block-bootstrap tests ===")
print(diff_tbl.to_string())
diff_tbl.to_csv("../outputs/nb_tbl_regime_difference_tests.csv")
print("\nAny row reading 'no' means that regime's correlation is NOT statistically")
print("distinguishable from the baseline, and the narrative should not lean on it.")

## 7. Return frequency: does the horizon change the answer?

Every result so far uses **daily** returns. The stock-bond correlation literature this
study is replicating — Campbell, Pflueger & Viceira (2020), Portelli & Roncalli (2024),
and the IMF note — works at monthly or quarterly horizons.

That choice is not cosmetic. Daily returns carry microstructure noise and
non-synchronous pricing: the Ibovespa close, the Tesouro PU reference price and the
CDI accrual are not struck at the same instant. Both effects attenuate correlation
toward zero. If the macro co-movement the paper is about lives at business-cycle
frequency, a daily estimate will systematically understate it.

In [ ]:
rows = []
for lab, rule in [("daily", None), ("weekly", "W-FRI"), ("monthly", "ME"), ("quarterly", "QE")]:
    for a, b in pairs:
        x = master[[a, b]].dropna()
        if rule:
            x = x.resample(rule).sum()          # log returns aggregate by summing
            x = x[(x != 0).any(axis=1)]
        r = corr_with_ci(x[a], x[b])
        rows.append({"Frequency": lab, "Pair": pair_labels[(a,b)], "n": r["n"],
                     "rho": round(r["rho"], 3),
                     "95% CI": f"[{r['lo']:+.3f},{r['hi']:+.3f}]",
                     "sig": "*" if r["sig"] else ""})

freq_tbl = pd.DataFrame(rows)
wide = freq_tbl.pivot(index="Pair", columns="Frequency", values="rho")[
    ["daily","weekly","monthly","quarterly"]]
print("=== Ibovespa x bond correlation by return frequency ===")
print(wide.to_string())
print("\n=== With confidence intervals ===")
for _, r in freq_tbl.iterrows():
    print(f"  {r['Frequency']:<10} {r['Pair']:<20} n={r['n']:>5}  "
          f"rho={r['rho']:+.3f}  {r['95% CI']}{r['sig']}")
freq_tbl.set_index(["Frequency","Pair"]).to_csv("../outputs/nb_tbl_frequency_robustness.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))
for pair_lbl in wide.index:
    ax.plot(range(4), wide.loc[pair_lbl], marker="o", lw=1.8, label=pair_lbl)
ax.set_xticks(range(4)); ax.set_xticklabels(["daily","weekly","monthly","quarterly"])
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_ylabel("Pearson rho with Ibovespa")
ax.set_title("Stock-bond correlation rises with the return horizon\n"
             "Daily estimates understate the macro co-movement", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/fig_frequency_robustness.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Forbes-Rigobon: is the crisis correlation surge real?

This is the test the crisis-correlation claim stands or falls on.

Forbes & Rigobon (2002) showed that a measured correlation is conditional on market
volatility: when the variance of the source market rises, the sample correlation is
**mechanically biased upward** even if the underlying propagation mechanism is
completely unchanged. Applying that correction, they found the apparent correlation
surges during the 1997 Asian crisis, the 1994 Mexican devaluation and the 1987 crash
largely disappeared — the markets were not more connected, only more volatile.

Brazilian crisis windows have equity variance 5–10x the calm level, so this correction
is not optional here. The adjusted estimate is

$$\\rho^* = \\frac{\\rho_c}{\\sqrt{1 + \\delta(1 - \\rho_c^2)}}, \\qquad
  \\delta = \\frac{\\sigma^2_{crisis}}{\\sigma^2_{calm}} - 1$$

**"Contagion"** means ρ* still exceeds the calm-period correlation. Otherwise what
looks like a correlation spike is only **interdependence** — the same relationship,
observed through a higher-variance lens.

In [ ]:
from metrics import forbes_rigobon

calm = master[master["crisis"] == "Calm"]
assert len(calm) > 1000, "calm window empty — check the crisis label"

rows = []
for cname, (s, e) in CRISES.items():
    k = master[(master.index >= s) & (master.index <= e)]
    for col in ["ntnb", "ltn", "ntnf"]:
        o = forbes_rigobon(calm["ibov"], calm[col], k["ibov"], k[col])
        rows.append({"Crisis": cname, "Bond": LABELS[col], "n": o["n_crisis"],
                     "rho calm": round(o["rho_calm"], 3),
                     "rho crisis (raw)": round(o["rho_crisis"], 3),
                     "equity vol ratio": round(1 + o["delta"], 2),
                     "rho adjusted": round(o["rho_adj"], 3),
                     "verdict": {True: "contagion", False: "interdependence",
                                 None: "n/a"}[o["contagion"]]})

fr_tbl = pd.DataFrame(rows).set_index(["Crisis", "Bond"])
print("=== Forbes-Rigobon volatility-adjusted crisis correlations ===")
print(fr_tbl.to_string())
fr_tbl.to_csv("../outputs/nb_tbl_forbes_rigobon.csv")

sub = fr_tbl.xs("NTN-B 5y", level="Bond")
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(sub)); w = 0.27
ax.bar(x - w, sub["rho calm"],         w, label="Calm period",   color="#1f77b4")
ax.bar(x,     sub["rho crisis (raw)"], w, label="Crisis (raw)",  color="#d62728")
ax.bar(x + w, sub["rho adjusted"],     w, label="Crisis (vol-adjusted)", color="#2ca02c")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(sub.index, fontsize=9)
ax.set_ylabel("Pearson rho (Ibovespa x NTN-B)")
ax.set_title("Most of the crisis correlation spike is a volatility artefact\n"
             "Green above blue = genuine contagion; green below = interdependence only",
             fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/fig_forbes_rigobon.png", dpi=150, bbox_inches="tight")
plt.show()

## ✅ Notebook 03 complete

The output above replaces four claims that the earlier version of this study asserted
without testing. Read the printed values rather than this cell, but note what each
section is now capable of falsifying:

| Section | What it tests | Why it matters |
|---------|---------------|----------------|
| 5 | Regime correlations **with confidence intervals** | A regime whose interval spans zero cannot be described as diversifying or not diversifying. |
| 6 | Whether regimes **differ from each other** | Ranking six point estimates is not evidence of regime change if the differences are inside the noise band. |
| 7 | Whether the answer **survives the return horizon** | Daily correlations are attenuated by microstructure noise; the literature this replicates uses monthly data. |
| 8 | Whether crisis correlation spikes **survive the volatility adjustment** | Forbes-Rigobon is cited in the methodology of this study; applying it is what makes the crisis claim testable rather than mechanical. |

**Outputs:** `fig_rolling_correlation.png`, `fig_conditional_correlations.png`,
`fig_frequency_robustness.png`, `fig_forbes_rigobon.png`,
`tbl_regime_correlations.csv`, `tbl_regime_difference_tests.csv`,
`tbl_frequency_robustness.csv`, `tbl_forbes_rigobon.csv`

**Next:** `04_dcc_garch.ipynb` — formal time-varying correlation via DCC-GARCH